In [34]:
from data import load_data
#from utils import plot_pool
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import numpy as np
from scipy.spatial.distance import cdist

In [37]:
data = load_data()
print(f"Train X len: {len(data["train"]["X"])}")
print(f"Train y len: {len(data["train"]["y"])}\n")

print(f"Pool X len: {len(data["pool"]["X"])}")
print(f"Pool y len: {len(data["pool"]["y"])}\n")

print(f"Test X len: {len(data["test"]["X"])}")
print(f"Test y len: {len(data["test"]["y"])}\n")


For digit 0 we select 592
For digit 1 we select 674
For digit 2 we select 595
For digit 3 we select 613
For digit 4 we select 584
For digit 5 we select 54
For digit 6 we select 59
For digit 7 we select 62
For digit 8 we select 58
For digit 9 we select 59
Train X len: 20
Train y len: 20

Pool X len: 3330
Pool y len: 3330

Test X len: 10000
Test y len: 10000



In [ ]:
#Helper functions from solution
def evaluate_uncertainty(prob, strategy):

    if strategy == 'least confident':
        res = 1 - prob.max(1)
    elif strategy == 'margin':
        ix = np.arange(len(prob))
        p2, p1 = prob.argsort(1)[:, -2:].T
        res = 1 - (prob[ix, p1] - prob[ix, p2])
    elif strategy == 'entropy':
        res = - np.sum(prob * np.log2(prob), axis=1)
    else:
        raise ValueError
    return res

def update_data(data, idx):
    """Update of the data dictionary from `prepare_data` by moving the data
    point with index `idx` from the pool to the training set."""
    data['train']['X'] = np.append(data['train']['X'], np.atleast_2d(data['pool']['X'][idx]), axis=0)
    data['train']['y'] = np.append(data['train']['y'], np.atleast_1d(data['pool']['y'][idx]), axis=0)    
    data['pool']['X'] = np.delete(data['pool']['X'], idx, axis=0)
    data['pool']['y'] = np.delete(data['pool']['y'], idx, axis=0)

def fit_model(paradigm, strategy, n_init, n_iterations, use_classes=None, use_features=None, plot=False):
    
    scores = np.zeros(n_iterations)
    model = LogisticRegression(penalty='l2', C=1e1, solver='liblinear', warm_start=True)



    for i in range(n_iterations):

        model = model.fit(data['train']['X'], data['train']['y'])


        prob = model.predict_proba(data['pool']['X'])
        scores[i] = model.score(data['test']['X'], data['test']['y'])

    
        if i < n_iterations:
            if paradigm == 'active learning':
                uncertainty = evaluate_uncertainty(prob, strategy)
                if strategy in ('least confident', 'entropy'):
                    idx = uncertainty.argmax()
                elif strategy == 'maximum margin':
                    idx = uncertainty.argmin()
                else:
                    raise ValueError
            elif paradigm == 'random':
                uncertainty = None
                idx = np.random.choice(np.arange(len(data['pool']['X'])))
            else:
                raise ValueError
            if plot:
                #plot_pool(data, idx, uncertainty)
                print("Get plottet")
          
            update_data(data, idx)
    return scores

def update_data(data, idx):
   
    data['train']['X'] = np.append(data['train']['X'], np.atleast_2d(data['pool']['X'][idx]), axis=0)
    data['train']['y'] = np.append(data['train']['y'], np.atleast_1d(data['pool']['y'][idx]), axis=0)    
    data['pool']['X'] = np.delete(data['pool']['X'], idx, axis=0)
    data['pool']['y'] = np.delete(data['pool']['y'], idx, axis=0)
